# Cardiac Surgery Analysis

This notebook provides reproducible exploratory analysis and a baseline classifier for the cardiac-surgery dataset. It does not ship with data and must not be treated as a clinical decision system.

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

RANDOM_STATE = 42
DATA_ENV_VAR = "CARDIAC_SURGERY_DATA"
BANNER_ENV_VAR = "CARDIAC_SURGERY_BANNER"

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")


In [ ]:
# The banner is optional. Paths supplied through environment variables may be absolute
# or relative to the directory from which Jupyter was started.
banner_path = Path(os.environ.get(BANNER_ENV_VAR, "data/Cardiac-Surgery-banner.webp")).expanduser()
if banner_path.exists():
    image = mpimg.imread(banner_path)
    plt.figure(figsize=(10, 3))
    plt.imshow(image)
    plt.axis("off")
    plt.title("Cardiac Surgery")
    plt.show()
else:
    print(f"Optional banner not found; continuing without it: {banner_path}")


In [ ]:
data_path = Path(os.environ.get(DATA_ENV_VAR, "data/cardiac-surgery.csv")).expanduser()
if not data_path.is_file():
    raise FileNotFoundError(
        f"Cardiac-surgery CSV was not found at {data_path}. "
        f"Download/obtain the licensed dataset and set {DATA_ENV_VAR} "
        "to its absolute path before running this notebook."
    )

data = pd.read_csv(data_path)
print(f"Loaded {len(data):,} rows and {len(data.columns):,} columns from {data_path}")


In [ ]:
required_columns = {
    "Region",
    "Procedure",
    "Year of Hospital Discharge",
    "Number of Cases",
    "Number of Deaths",
    "Expected Mortality Rate",
    "Observed Mortality Rate",
    "Comparison Results",
}
missing_columns = sorted(required_columns.difference(data.columns))
if missing_columns:
    raise ValueError(
        "The input dataset is missing required columns: " + ", ".join(missing_columns)
    )

# These identifiers are not used by the analysis. Ignore them if present so the
# notebook also works with already-anonymised extracts.
data = data.drop(columns=["Facility ID", "Hospital Name", "Detailed Region"], errors="ignore")


In [ ]:
# Parse the year without assuming a single source format. Invalid values remain NaN
# and are imputed inside the training-only model pipeline below.
year_column = "Year of Hospital Discharge"
year_as_date = pd.to_datetime(data[year_column], errors="coerce").dt.year
year_as_number = pd.to_numeric(data[year_column], errors="coerce")
data[year_column] = year_as_date.fillna(year_as_number)

print("Missing values by column:")
display(data.isna().sum().sort_values(ascending=False).to_frame("missing"))


## Exploratory analysis

The summaries below are descriptive only. Missing values are not filled globally for modeling; the classifier fits its imputers on the training split only.

In [ ]:
display(data.head())
display(data.describe(include="all").transpose())


In [ ]:
numeric_data = data.select_dtypes(include="number")
if not numeric_data.empty:
    plt.figure(figsize=(10, 5))
    sns.heatmap(numeric_data.corr(), annot=True, cmap="coolwarm", center=0)
    plt.title("Numeric feature correlations")
    plt.tight_layout()
    plt.show()


In [ ]:
procedure_summary = data.groupby("Procedure")[["Number of Cases", "Number of Deaths"]].mean()
display(procedure_summary)
procedure_summary.plot(kind="bar", figsize=(10, 5))
plt.title("Average outcomes by procedure")
plt.xlabel("Procedure")
plt.ylabel("Average value")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
region_summary = data.groupby("Region")[["Number of Cases", "Number of Deaths"]].mean()
display(region_summary)
region_summary.plot(kind="bar", figsize=(10, 5))
plt.title("Average outcomes by region")
plt.xlabel("Region")
plt.ylabel("Average value")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
region_year_counts = (
    data.groupby(["Region", year_column])
    .size()
    .rename("case_count")
    .reset_index()
)
display(region_year_counts.sort_values("case_count", ascending=False).head(20))


In [ ]:
observed_mortality_rate = data.groupby("Procedure")["Observed Mortality Rate"].mean().sort_values(ascending=False)
display(observed_mortality_rate.to_frame("mean_observed_mortality_rate"))


In [ ]:
mortality_rates = data.groupby("Procedure")[["Observed Mortality Rate", "Expected Mortality Rate"]].mean()
display(mortality_rates)
mortality_rates.plot(kind="bar", figsize=(10, 5), width=0.8)
plt.title("Mean observed vs expected mortality rate by procedure")
plt.xlabel("Procedure")
plt.ylabel("Mortality rate")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


## Leakage-safe baseline classification

The target is **Comparison Results**. Categorical values are one-hot encoded with unknown-category handling, and numeric missing values are median-imputed inside a `Pipeline`. Therefore, preprocessing is fitted on `X_train` only. The split is stratified when every class has at least two rows; otherwise the notebook makes the limitation explicit by using an unstratified split.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

model_features = [
    "Region",
    "Procedure",
    year_column,
    "Number of Cases",
    "Number of Deaths",
    "Expected Mortality Rate",
]
target_column = "Comparison Results"
model_data = data[model_features + [target_column]].dropna(subset=[target_column]).copy()
model_data[target_column] = model_data[target_column].astype(str)

if len(model_data) < 10:
    raise ValueError("At least 10 labeled rows are required for a meaningful train/test split.")
if model_data[target_column].nunique() < 2:
    raise ValueError("The target must contain at least two classes for classification.")

X = model_data[model_features]
y = model_data[target_column]
stratify = y if y.value_counts().min() >= 2 else None
if stratify is None:
    print("Warning: at least one target class has one row; using an unstratified split.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=stratify
)

numeric_features = [year_column, "Number of Cases", "Number of Deaths", "Expected Mortality Rate"]
categorical_features = ["Region", "Procedure"]
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([("imputer", SimpleImputer(strategy="median"))]),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)
model = Pipeline(
    [
        ("preprocess", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


In [ ]:
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_test, y_pred):.3f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

labels = model.named_steps["classifier"].classes_
confusion = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=labels),
    index=[f"actual: {label}" for label in labels],
    columns=[f"predicted: {label}" for label in labels],
)
display(confusion)


## Reproducibility and limitations

- The split seed is fixed at `RANDOM_STATE = 42`; metrics still depend on the supplied data and package versions.
- No metric is reported in this repository because the dataset is intentionally absent.
- This baseline is for educational analysis only and is not validated for clinical use.
